# global_config\nGenerated from 00_common/global_config.py for Databricks notebook import.\n

In [ ]:
"""Global configuration loader with environment variable substitution."""\n\nfrom __future__ import annotations\n\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n_ENV_PATTERN = re.compile(r"\$\{([A-Z0-9_]+)(?::([^}]*))?\}")\n\n\ndef _substitute_env(value: str) -> str:\n    def replacer(match: re.Match[str]) -> str:\n        key = match.group(1)\n        default_value = match.group(2) if match.group(2) is not None else ""\n        return os.getenv(key, default_value)\n\n    return _ENV_PATTERN.sub(replacer, value)\n\n\ndef _resolve_obj(obj: Any) -> Any:\n    if isinstance(obj, str):\n        return _substitute_env(obj)\n    if isinstance(obj, dict):\n        return {k: _resolve_obj(v) for k, v in obj.items()}\n    if isinstance(obj, list):\n        return [_resolve_obj(v) for v in obj]\n    return obj\n\n\ndef load_global_config(config_path: str) -> dict:\n    path = Path(config_path)\n    if not path.exists():\n        raise FileNotFoundError(\n            f"Global config not found: {path}. "\n            "Set --global-config to the correct path or check your deployment."\n        )\n    try:\n        with path.open("r", encoding="utf-8") as f:\n            payload = yaml.safe_load(f) or {}\n    except yaml.YAMLError as exc:\n        raise ValueError(f"Invalid YAML in global config {path}: {exc}") from exc\n    # Add default for lifecycle log table if not present\n    if "lifecycle_log_table" not in payload:\n        payload["lifecycle_log_table"] = "audit.entity_lifecycle_log"\n    return _resolve_obj(payload)\n\n\ndef require_config_keys(config: dict, required_dotted_paths: list[str]) -> None:\n    """Raise ValueError listing every dotted key path that is missing or empty.\n\n    Example::\n\n        require_config_keys(cfg, ["audit.pipeline_runs_table", "organization.framework_name"])\n    """\n    missing: list[str] = []\n    for dotted in required_dotted_paths:\n        parts = dotted.split(".")\n        node: Any = config\n        for part in parts:\n            if not isinstance(node, dict) or part not in node:\n                missing.append(dotted)\n                break\n            node = node[part]\n        else:\n            if not node:  # catches empty string / None / 0 / []\n                missing.append(dotted)\n    if missing:\n        raise ValueError(f"Missing required global config values: {missing}")\n